# Módulo 2 — Punto 2.2: Morfología matemática

Toda la lógica del cálculo está en `src/modulo2/morfologia.py`; este notebook se encarga de llamar a los módulos y visualizar los resultados.

## 0. Configuración del entorno e importaciones
Se configura la ruta raíz del repositorio y se importan las librerías necesarias.

In [ ]:
# --- Imports de la librería estándar ---
import sys                 
from pathlib import Path   

# --- Configurar la raíz del repositorio ---
# Path.cwd() = carpeta desde donde corre el notebook (normalmente notebooks/).
# Subimos carpetas hasta encontrar config.yaml: esa es la raíz.
# Así funciona tanto si VS Code abre el notebook desde notebooks/ como desde la raíz.
raiz = Path.cwd()
while not (raiz / "config.yaml").exists():
    if raiz.parent == raiz:          # llegamos a "/" sin encontrarlo
        raise FileNotFoundError("No encontré config.yaml en ninguna carpeta superior.")
    raiz = raiz.parent

# Insertamos la raíz al principio de sys.path (posición 0 = máxima prioridad).
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

# --- Imports de terceros y propios
import numpy as np                 
import matplotlib.pyplot as plt    
import matplotlib.patches as patches  
import skimage as ski



from src.config import cargar_config
from src.modulo1 import carga_muestreo as cm   
from src.modulo1 import calidad_filtrado as cf
from src.modulo2 import segmentacion as seg
from src.modulo2 import morfologia as morf

from IPython.display import display, HTML
display(HTML("<style>.output_png { display: flex; justify-content: center; }</style>"))




## 1. Carga de Datos y Configuración Global
Se leen los parámetros desde `config.yaml`, se definen las rutas de salida e inicializan las imágenes de trabajo.

In [2]:
# Cargar configuración global
cfg = cargar_config(raiz / "config.yaml")

# Directorio de salida de resultados
carpeta_res = raiz / cfg["salida"]["carpeta_resultados"]
carpeta_res.mkdir(parents=True, exist_ok=True)

# Carga de imagen y extracción de la ROI
img = cm.obtener_frame(cfg, raiz)
roi = cm.recortar_roi(img, cfg["roi"]) 
roi_grises = cf.conv_grises(roi) 




print(f"Resultados en: {carpeta_res}")

Resultados en: c:\Users\Camila\Desktop\Guia_Imagenes\Guia-de-imagenes-biomedicas-G9-PSIB\results\modulo1


In [ ]:
# Radio aproximado de la pupila en píxeles
r = 30  # Reemplazar por el radio de tu ROI

# Máscara original de Watershed
mascara_original = seg.segmentar_watershed(roi_)

# Elemento estructurante
ee = morf.elemento(r, factor=0.05)

# Aplicar operaciones y combinaciones
resultados = {
    "Watershed original": morf._u8(mascara_original),

    "Solo erosión": morf.erosion(mascara_original, ee),

    "Solo dilatación": morf.dilatacion(mascara_original, ee),

    "Solo apertura": morf.apertura(mascara_original, ee),

    "Solo cierre": morf.cierre(mascara_original, ee),

    "Cierre + apertura": morf.apertura(
        morf.cierre(mascara_original, ee), ee
    ),

    "Apertura + cierre": morf.cierre(
        morf.apertura(mascara_original, ee), ee
    ),
}

# Mostrar los resultados
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, (nombre, mascara) in zip(axes.flat, resultados.items()):
    ax.imshow(mascara, cmap="gray", vmin=0, vmax=255)
    ax.set_title(nombre)
    ax.axis("off")

# Ocultar el subplot sobrante
axes.flat[-1].axis("off")

plt.tight_layout()
plt.show()